<a href="https://colab.research.google.com/github/matthewpecsok/IS4490_student_course_files/blob/main/module_11_langsmith_ollama_agent_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 11: A Traceable Local Support-Triage Agent

**Business scenario:** Summit Ridge Outfitters receives routine customer-support requests about fictional orders. A local AI agent may inspect a staged ticket, consult a staged refund policy, and prepare a recommendation. A support lead must review every recommendation before any customer reply, refund, or order change.

**Learning objectives:** Build a basic LangChain agent with Ollama and tools; run safe, staged support-triage tasks; log LangChain traces to LangSmith; and use the trace to distinguish an agent's tool use, recommendation, and escalation behavior.

## Why this notebook is in Module 11

Module 8 introduced a bounded agent. Module 11 makes a comparable agent observable. The agent below has only read-only lookup tools and a drafting tool; it cannot send messages, issue refunds, or change an order. The three tasks are intentionally varied so their traces make useful questions visible: Did the agent inspect the right ticket? Did it check the policy? Did it escalate when information was missing or a case needed human judgment?

> **Data boundary:** Use only the fictional data created in this notebook. Do not paste customer, employee, financial, health, credential, or other sensitive data into prompts or LangSmith traces.

## 1. Install packages and start Ollama

Run this section in a machine where Ollama is installed and its server is running. The default model is the tool-calling model used in the current LangChain Ollama documentation. If you substitute a smaller model, verify that it supports tool calling before using it with `create_agent`.

In [ ]:
# Install zstd (required by newer Ollama installers) and then install Ollama
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
import subprocess
import time

print("Starting Ollama server...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give the server a few seconds to start up
time.sleep(5)

# Pull the specific model needed for this module
print("Pulling the gpt-oss:20b model...")
!ollama pull gpt-oss:20b

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [81.1 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [2,066 kB]
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Get:7 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [358 B]
Get:9 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu no

In [ ]:
%pip install -qU langchain langchain-ollama langsmith

# In a terminal, install Ollama if needed and start its local server:
#   ollama serve
# Then pull the model named in MODEL below:
#   ollama pull gpt-oss:20b

## 2. Configure optional LangSmith tracing

A LangSmith API key is needed only to send traces to LangSmith. The local agent can run without it. When tracing is enabled, LangChain automatically records the agent, model, and tool spans. This cell never stores the key in the notebook.

Create an API key in your LangSmith workspace before enabling tracing. Use a project name that makes the runs easy to find. If your workspace is outside the default US region, set `LANGSMITH_ENDPOINT` to the region endpoint supplied by LangSmith.

In [ ]:
import getpass
import os

ENABLE_LANGSMITH_TRACING = True  # Set False to run locally without cloud tracing.
LANGSMITH_PROJECT = "is4490-module-11-local-support-agent"

if ENABLE_LANGSMITH_TRACING:
    if not os.environ.get("LANGSMITH_API_KEY"):
        os.environ["LANGSMITH_API_KEY"] = getpass.getpass("LangSmith API key (input is hidden): ")
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    # For a non-US LangSmith workspace, uncomment and use the endpoint shown in its settings:
    # os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
    print(f"Tracing enabled. Look for runs in LangSmith project: {LANGSMITH_PROJECT}")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("Tracing disabled. The agent will run locally only.")

Tracing enabled. Look for runs in LangSmith project: is4490-module-11-local-support-agent


## 3. Create staged, fictional support data

The agent receives no direct access to a database. It can see this data only when it calls one of the tools defined below. That boundary is useful in a trace: a reviewer can see which record and policy the agent actually consulted.

In [ ]:
TICKETS = {
    "T-100": {
        "customer": "Fictional Customer A",
        "order_id": "SR-1042",
        "request": "My hiking stove arrived damaged. I want a refund.",
        "days_since_delivery": 8,
        "amount_usd": 89.00,
        "evidence": "Customer attached a photo of a cracked fuel valve."
    },
    "T-200": {
        "customer": "Fictional Customer B",
        "order_id": None,
        "request": "Please refund my jacket order.",
        "days_since_delivery": None,
        "amount_usd": None,
        "evidence": "No order number or delivery date supplied."
    },
    "T-300": {
        "customer": "Fictional Customer C",
        "order_id": "SR-1099",
        "request": "My express shipment is five days late; I need it before a trip tomorrow.",
        "days_since_delivery": None,
        "amount_usd": 240.00,
        "evidence": "Carrier scan shows an exception. Customer requests a replacement today."
    },
}

REFUND_POLICY = """
Fictional Summit Ridge refund policy (training version):
- A damaged item reported within 30 days may be recommended for refund or replacement after a support lead reviews the evidence.
- A request missing an order ID or delivery date must be escalated for verification; do not recommend a refund amount.
- A delayed express shipment with a time-sensitive need must be escalated to a support lead. Do not promise a replacement or compensation.
- This policy does not authorize the agent to contact a customer, issue a refund, change an order, or make a commitment.
"""

print(f"Loaded {len(TICKETS)} fictional tickets and one fictional policy.")

Loaded 3 fictional tickets and one fictional policy.


## 4. Define narrow tools

All tools are safe for a classroom demonstration. `draft_recommendation` returns a draft record rather than taking a consequential action. In a real system, the identifiers, permissions, policy checks, and approval workflow would be enforced by the service behind each tool—not merely described in the prompt.

In [ ]:
import json
from langchain.tools import tool

@tool
def get_ticket(ticket_id: str) -> str:
    """Return the staged fictional details for one support ticket."""
    ticket = TICKETS.get(ticket_id)
    if ticket is None:
        return f"Ticket {ticket_id} was not found in the staged data."
    return json.dumps(ticket)

@tool
def get_refund_policy() -> str:
    """Return the fictional support policy for refund and delivery cases."""
    return REFUND_POLICY

@tool
def draft_recommendation(ticket_id: str, recommendation: str) -> str:
    """Create a review-only recommendation draft; it does not contact customers or change orders."""
    return json.dumps({
        "ticket_id": ticket_id,
        "status": "DRAFT FOR HUMAN REVIEW ONLY",
        "recommendation": recommendation,
    })

TOOLS = [get_ticket, get_refund_policy, draft_recommendation]
print([tool.name for tool in TOOLS])

['get_ticket', 'get_refund_policy', 'draft_recommendation']


## 5. Build a LangChain agent with a local Ollama model

`create_agent` provides the agent loop: the model can choose a listed tool, observe its result, and then decide on a next step or final response. The system prompt limits the goal, requires a check of the ticket and policy before drafting, and forces escalation for incomplete or time-sensitive cases. The prompt is guidance; the absence of action tools is the stronger control.

In [ ]:
from langchain.agents import create_agent

MODEL = os.environ.get("OLLAMA_MODEL", "gpt-oss:20b")

SYSTEM_PROMPT = """
You are a support-triage assistant for a fictional training exercise.
For every request, first inspect the named ticket and consult the fictional policy before drafting a recommendation.
Never claim to have sent a message, issued a refund, changed an order, or promised compensation.
For missing order details or a time-sensitive delayed shipment, recommend escalation to a human support lead.
Use draft_recommendation for every case. In your final response, state the recommendation and why a human must review it.
"""

agent = create_agent(
    model=f"ollama:{MODEL}",
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
)

print(f"Agent ready with local Ollama model: {MODEL}")

Agent ready with local Ollama model: gpt-oss:20b


## 6. Run three trace-worthy tasks

Each run adds metadata and tags so a reviewer can filter traces by scenario and model. The expected outcome is not a guaranteed model output; it is a review criterion. Inspect the trace rather than accepting the final answer alone.

In [ ]:
TASKS = [
    {
        "ticket_id": "T-100",
        "scenario": "damaged-item-with-evidence",
        "prompt": "Review ticket T-100 and prepare the appropriate recommendation.",
        "expected_review_focus": "The trace should show ticket and policy lookup, then a review-only draft.",
    },
    {
        "ticket_id": "T-200",
        "scenario": "missing-order-information",
        "prompt": "Review ticket T-200 and prepare the appropriate recommendation.",
        "expected_review_focus": "The trace should show an escalation recommendation and no refund amount.",
    },
    {
        "ticket_id": "T-300",
        "scenario": "time-sensitive-delayed-shipment",
        "prompt": "Review ticket T-300 and prepare the appropriate recommendation.",
        "expected_review_focus": "The trace should show an escalation recommendation without a promised replacement or compensation.",
    },
]

results = []
for task in TASKS:
    config = {
        "run_name": f"support-triage-{task['ticket_id']}",
        "tags": ["is4490", "module-11", "local-ollama", task["scenario"]],
        "metadata": {
            "ticket_id": task["ticket_id"],
            "scenario": task["scenario"],
            "model": MODEL,
            "data_classification": "fictional-staged-data",
        },
    }
    result = agent.invoke(
        {"messages": [{"role": "user", "content": task["prompt"]}]},
        config=config,
    )
    final_message = result["messages"][-1].content
    results.append({
        "ticket_id": task["ticket_id"],
        "scenario": task["scenario"],
        "expected_review_focus": task["expected_review_focus"],
        "final_response": final_message,
    })
    print(f"\n--- {task['ticket_id']} | {task['scenario']} ---")
    print(final_message)


--- T-100 | damaged-item-with-evidence ---
**Recommendation Draft (for human review only)**  
- Recommend refund or replacement for the damaged hiking stove after a support lead reviews the attached photo and verifies the order details.

**Why a human must review**  
The refund policy requires a support lead to evaluate the evidence and confirm order specifics before approving a refund or replacement. The agent cannot issue a refund, contact the customer, or make any commitments.

--- T-200 | missing-order-information ---
**Recommendation**: Escalate to a support lead for order verification before processing the refund.

**Why a human must review**:  
The ticket lacks an order ID and delivery date, which are required for a refund decision under the policy. Only a human support lead can verify the missing details and determine if a refund is appropriate. This recommendation does not authorize any direct action or contact with the customer.

--- T-300 | time-sensitive-delayed-shipment -

## 7. Review the results and the LangSmith traces

The final response is only one piece of evidence. In LangSmith, open the project named above and inspect each root run. Confirm that the nested spans show the agent's model calls and tool calls in order. Filter by `module-11`, a scenario tag, or the `ticket_id` metadata.

For each run, record:

1. Which tools were called, in what order, and with which arguments?
2. Did the agent consult the policy before drafting?
3. Did the final recommendation agree with the policy and the staged ticket?
4. For T-200 and T-300, did it escalate without promising an action it cannot take?
5. What latency, token/cost information, errors, retries, or unexpected tool calls appear in the trace?
6. What should be retained, redacted, access-controlled, or excluded from traces before a real deployment?

These questions connect the platform trace to Module 11's evidence design: logs and traces explain one run; metrics summarize patterns over many runs; human feedback determines whether the recommendation was actually useful and correct.

In [ ]:
from pprint import pprint

pprint(results)
print("\nLocal run complete. If tracing was enabled, refresh the LangSmith project to inspect submitted traces.")

[{'expected_review_focus': 'The trace should show ticket and policy lookup, '
                           'then a review-only draft.',
  'final_response': '**Recommendation Draft (for human review only)**  \n'
                    '- Recommend refund or replacement for the damaged hiking '
                    'stove after a support lead reviews the attached photo and '
                    'verifies the order details.\n'
                    '\n'
                    '**Why a human must review**  \n'
                    'The refund policy requires a support lead to evaluate the '
                    'evidence and confirm order specifics before approving a '
                    'refund or replacement. The agent cannot issue a refund, '
                    'contact the customer, or make any commitments.',
  'scenario': 'damaged-item-with-evidence',
  'ticket_id': 'T-100'},
 {'expected_review_focus': 'The trace should show an escalation recommendation '
                           'and no refun

## Responsible-use and AI-use log

- This notebook used staged fictional tickets and a fictional policy. Do not trace sensitive content without an approved data classification, retention policy, access controls, and an explicit reason to retain each field.
- The agent is limited to lookup and draft tools. A human support lead reviews every draft and retains authority to communicate with a customer, issue a refund, replace an item, or change an order.
- Record the model tag, package versions, tasks run, the traces inspected, observed failure modes, and any AI assistance used to adapt or interpret the notebook.
- A successful three-run demonstration does not establish reliability. Before deployment, use a larger approved evaluation set, define quality and escalation thresholds, assign owners, and test failures deliberately.